# RM/PM EDA And Time-Series Evidence

Examines missingness, demand distributions, intermittency, seasonality, autocorrelation, scale variance and concentration before modeling.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("Ai miroservices/modeling/project_operational_baseline").resolve()
OUT = ROOT / "outputs"
sns.set_theme(style="whitegrid")

In [ ]:
demand = pd.read_csv(OUT / 'demand_history.csv.gz', parse_dates=['month'])
quality = pd.DataFrame({'missing': demand.isna().sum(), 'missing_pct': demand.isna().mean()*100})
display(quality[quality.missing.gt(0)])
display(demand.groupby('material_type').demand_units.describe(percentiles=[.5,.9,.95,.99]))

In [ ]:
summary = demand.groupby(['material_id','material_code','material_type']).demand_units.agg(['mean','std',lambda s:(s==0).mean()]).reset_index()
summary.columns=['material_id','material_code','material_type','mean','std','zero_share']
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.histplot(data=summary,x='mean',hue='material_type',log_scale=True,ax=axes[0])
sns.scatterplot(data=summary,x='mean',y='std',hue='material_type',ax=axes[1]); plt.show()

In [ ]:
monthly = demand.groupby(['material_type','month']).demand_units.sum().reset_index()
monthly['month_of_year']=monthly.month.dt.month
season = monthly.groupby(['material_type','month_of_year']).demand_units.mean().reset_index()
season['index']=season.demand_units/season.groupby('material_type').demand_units.transform('mean')
sns.lineplot(data=season,x='month_of_year',y='index',hue='material_type',marker='o'); plt.axhline(1,color='black',lw=1); plt.show()